In [3]:
!pip install -r /home/jovyan/requirements.txt -q

print("✅ Toutes les dépendances installées")

✅ Toutes les dépendances installées


In [4]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *
import pandas as pd
import folium
import json

spark = SparkSession.builder \
    .appName("TaaSim-Remapping") \
    .config("spark.driver.memory", "4g") \
    .config("spark.jars.packages",
            "org.apache.hadoop:hadoop-aws:3.3.4,com.amazonaws:aws-java-sdk-bundle:1.12.262") \
    .config("spark.hadoop.fs.s3a.endpoint", "http://minio:9000") \
    .config("spark.hadoop.fs.s3a.access.key", "taasim") \
    .config("spark.hadoop.fs.s3a.secret.key", "taasim123") \
    .config("spark.hadoop.fs.s3a.path.style.access", "true") \
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem") \
    .config("spark.hadoop.fs.s3a.connection.ssl.enabled", "false") \
    .getOrCreate()

print("✅ Spark prêt avec MinIO")

✅ Spark prêt avec MinIO


In [6]:
# Traite un seul mois à la fois
df = spark.read.parquet("s3a://raw/nyc-tlc/*.parquet", header=True, inferSchema=True)

print(f"Lignes : {df.count()}")



Lignes : 11198026


In [7]:
from pyspark.sql.functions import col, count, when


# Découverte de la data
print(f"\033[1;34m Découverte de la data \033[0m")

print("\033[1;37m Schema de la data \033[0m")
df.printSchema()
print(f"Nombre de lignes : {df.count()}")
print(f"Nombre de colonnes : {len(df.columns)}")

print("\033[1;37m Head of data \033[0m")
df.show(5)

print("\033[1;37m Description of data \033[0m")
df.describe().show()

print("\033[1;37m Calcul des valeurs nulles \033[0m")
df.select([
    count(when(col(c).isNull(), c)).alias(c)
    for c in df.columns
]).show()

print("\033[1;37m Nombre de doublons \033[0m")
total = df.count()
unique = df.dropDuplicates().count()
print(f"Doublons : {total - unique}")
print("-----------------------------------------")
print("fin du test")

 Découverte de la data 
 Schema de la data 
root
 |-- VendorID: integer (nullable = true)
 |-- tpep_pickup_datetime: timestamp_ntz (nullable = true)
 |-- tpep_dropoff_datetime: timestamp_ntz (nullable = true)
 |-- passenger_count: long (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- RatecodeID: long (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- PULocationID: integer (nullable = true)
 |-- DOLocationID: integer (nullable = true)
 |-- payment_type: long (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- extra: double (nullable = true)
 |-- mta_tax: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- tolls_amount: double (nullable = true)
 |-- improvement_surcharge: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- congestion_surcharge: double (nullable = true)
 |-- Airport_fee: double (nullable = true)
 |-- cbd_congestion_fee: double (nullable = true)

Nombre de lignes : 11198026
Nom

In [8]:
from pyspark.sql.functions import col

def clean_df(df):
    
    # VendorID valides
    df = df.filter(col('VendorID').isin([1, 2, 6, 7]))
    
    # RatecodeID valides
    df = df.filter(col('RatecodeID').isin([1, 2, 3, 4, 5, 6, 99]))
    
    # payment_type valides
    df = df.filter(col('payment_type').isin([0, 1, 2, 3, 4, 5, 6]))
    
    # store_and_fwd_flag valides
    df = df.filter(col('store_and_fwd_flag').isin(['Y', 'N']))
    
    # Distance réaliste
    df = df.filter((col('trip_distance') > 0.1) & (col('trip_distance') <= 100))
    
    # Fare positif et réaliste
    df = df.filter((col('fare_amount') > 0) & (col('fare_amount') <= 300))
    
    # Passagers valides
    df = df.filter((col('passenger_count') >= 1) & (col('passenger_count') <= 9))
    
    # Supprimer les nulls restants
    df = df.dropna()
    
    return df

# Appliquer le nettoyage
avant = df.count()
df_clean = clean_df(df)
apres = df_clean.count()

print(f"Avant nettoyage  : {avant}")
print(f"Après nettoyage  : {apres}")
print(f"Lignes supprimées: {avant - apres}")

Avant nettoyage  : 11198026
Après nettoyage  : 8548853
Lignes supprimées: 2649173
